In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model                                            # model load krne ke liye...
import pickle
import pandas as pd 
import numpy as np

In [3]:
## load the trained model , scaler pickle , onehot 

model = load_model('model.h5')                                                      #  Train model load krne ke liye...

## load  the encoder and scaler pickle file                                        # onehot encoder and scaler pickle file load krne ke liye...
with open('onehot_encoder_geo.pkl', 'rb') as file:
    label_encoder_geo = pickle.load(file)
    
with open('label_encoder_gender.pkl', 'rb') as file:                                # gender label encoder pickle file load krne ke liye...
    label_encoder_gender = pickle.load(file)
    
    
with open('scaler.pkl', 'rb') as file:                                                 # Scaler pickle file load krne ke liye...
    scaler = pickle.load(file)  

In [4]:
## Example input data for prediction...           pickle in used in processing dject :- Text ko Number me convert karna .....

input_data = {                                              ## INPUT DATA....
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}  

# input_data   in key value pair....

In [5]:
# One hot Encode 'Geography'

geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()

# column = 'GEOGRAPHY'
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df


c:\Users\harsh\miniconda3\envs\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
# combine one-hot encoded columns with input data......
# input_data  ko  input_df me store .......


input_df = pd.DataFrame([input_data])     ## 

input_df




,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [7]:

# Encode  the categorical variables Gender

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])                            # Gender column me kro divide into  binary(0 ,1)

input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [8]:
# fresh input df...

input_df = pd.DataFrame([input_data])

In [9]:
# Concatenation ........ Combine one-hot encoded columns with input data .....   TO give info to model together...
                                                                                        # append geography columns ..
                                                                                        
input_df = pd.concat([input_df.drop("Geography",axis=1), geo_encoded_df],axis =1)  

input_df

   

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
## Scaling the input data ...
# Gender ko numeric mein convert karo

input_df['Gender'] = label_encoder_gender.transform(
    input_df['Gender']
)

# Ab scaling karo
input_scaled = scaler.transform(input_df)

input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

## Predict  the Churn ..

In [11]:
## predict churn 

prediction  = model.predict(input_scaled)

prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step


array([[0.02374217]], dtype=float32)

In [12]:
# prediction probability ...

prediction_probability = prediction[0][0]


In [13]:
prediction_probability

np.float32(0.02374217)

In [14]:
if prediction_probability>0.5:
    print("the customer is likely to churn..")
    
else:
    print("the cusstomer is not likely to churn ")    

the cusstomer is not likely to churn 
